# 05 - Monte Carlo Strategy Simulation

## Goal

Evaluate strategy robustness from Notebook 04 with Monte Carlo simulation:
- NPV distribution,
- confidence intervals,
- downside metrics (VaR/CVaR),
- sensitivity to demand and risk shocks.

Outputs: simulation_summary.csv, simulation_sensitivity.csv

In [ ]:
import sys, warnings
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

ROOT = Path().cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from simulation import (
    run_scenario_pack, run_sensitivity,
    plot_npv_distributions, plot_sensitivity_heatmap,
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

df_demand = pd.read_csv(ROOT / 'data' / 'df_demand.csv')
df_alloc = pd.read_csv(ROOT / 'data' / 'allocation_recommended.csv')
df_opt = pd.read_csv(ROOT / 'data' / 'optimization_summary.csv')

print(f'df_demand: {df_demand.shape}')
print(f'df_alloc : {df_alloc.shape}')
print(f'df_opt   : {df_opt.shape}')

---
## 1. Simulation Setup

For each customer and scenario:
- offers are taken from `x_recommended` (lp_round export from Notebook 04),
- acceptances are simulated as Binomial(offers, p_accept_scenario),
- defaults are simulated as Binomial(accepted, pd_annual),
- payoff per accepted offer: `40 - default * LGD * EAD`.

Risk assumptions are aligned with previous notebooks: `LGD=0.55`, `EAD=0.5 * initial_loan`,
discounting at 19% annual rate over a 2-month horizon.

In [ ]:
SCENARIOS = ['base', 'stress', 'benign']
N_SIMS = 5000
SEED = 42

sim_summary, sims = run_scenario_pack(
    df_demand,
    df_alloc,
    scenarios=SCENARIOS,
    n_sims=N_SIMS,
    seed=SEED,
)

display(sim_summary.round(2))

---
## 2. NPV Distributions

Inspect full scenario distributions, not only mean NPV.

In [ ]:
plot_npv_distributions(sims)

for sc in SCENARIOS:
    s = sims[sc]['npv']
    print(f'{sc:<8} mean={s.mean():,.1f}  std={s.std():,.1f}  p05={s.quantile(0.05):,.1f}  p95={s.quantile(0.95):,.1f}')

---
## 3. Comparison vs Deterministic Optimization

Compare simulated mean NPV with Notebook 04 `expected_npv`
for the integer strategy (`lp_round`).

In [ ]:
det = df_opt[df_opt['strategy'] == 'lp_round'][['scenario', 'expected_npv']].copy()
cmp = sim_summary.merge(det, on='scenario', how='left')
cmp['delta_vs_expected'] = cmp['npv_mean'] - cmp['expected_npv']
cmp['delta_pct'] = cmp['delta_vs_expected'] / cmp['expected_npv'] * 100
display(cmp[['scenario', 'npv_mean', 'expected_npv', 'delta_vs_expected', 'delta_pct', 'var95', 'cvar95']].round(2))

---
## 4. Sensitivity: p_accept and PD

Stress test NPV under parameter shocks:
- p_accept: 0.90x / 1.00x / 1.10x
- PD:       0.90x / 1.00x / 1.10x

In [ ]:
sens = run_sensitivity(
    df_demand,
    df_alloc,
    scenario='base',
    p_accept_multipliers=(0.90, 1.00, 1.10),
    pd_multipliers=(0.90, 1.00, 1.10),
    n_sims=3000,
    seed=123,
)
display(sens.round(2))

plot_sensitivity_heatmap(sens, value_col='npv_mean')
plot_sensitivity_heatmap(sens, value_col='cvar95')

---
## 5. Sanity checks

Validation invariants:
- `var95 <= npv_mean`,
- `cvar95 <= var95`,
- stress should not outperform benign in mean NPV.

In [ ]:
assert (sim_summary['var95'] <= sim_summary['npv_mean']).all(), 'VaR95 above mean NPV'
assert (sim_summary['cvar95'] <= sim_summary['var95']).all(), 'CVaR95 above VaR95'

m = sim_summary.set_index('scenario')['npv_mean']
assert m['stress'] <= m['benign'], 'Stress unexpectedly better than Benign'

print('Sanity checks passed.')

---
## 6. Export

Save simulation aggregates and sensitivity table for final reporting.

In [ ]:
out_sim = ROOT / 'data' / 'simulation_summary.csv'
out_sens = ROOT / 'data' / 'simulation_sensitivity.csv'
sim_summary.to_csv(out_sim, index=False)
sens.to_csv(out_sens, index=False)

print(f'Saved: {out_sim}')
print(f'Saved: {out_sens}')
print(f'simulation_summary shape: {sim_summary.shape}')
print(f'simulation_sensitivity shape: {sens.shape}')